The goal of this notebook: to decode templates (to match arguments of templates with language, word, etc.)

In [ ]:
import pandas as pd
import importlib
from utils import langcodes_v2
importlib.reload(langcodes_v2)
from utils import langcodes_v2

from utils import langcodes_v1
importlib.reload(langcodes_v1)
from utils import langcodes_v1 as langcodes

In [18]:
df = pd.read_pickle('../ety_data.pkl')

In [19]:
df

,word,lang,ety,templates,etymology_number,num_templates,split,subpart
0,abaft,English,"From a- (“on”) + Middle English baft, baften, ...","[{'name': 'm', 'args': {'1': 'en', '2': 'a-', ...",0,10,-1,1
1,abaft,English,"From a- (“on”) + Middle English baft, baften, ...","[{'name': 'm', 'args': {'1': 'en', '2': 'a-', ...",0,10,-1,1
2,abraid,English,"From Middle English abraiden, abreiden (“to st...","[{'name': 'inh', 'args': {'1': 'en', '2': 'enm...",1,10,-1,1
3,abear,English,"From Middle English aberen, from Old English ā...","[{'name': 'inh', 'args': {'1': 'en', '2': 'enm...",0,10,-1,1
4,abear,English,"From Middle English aberen, from Old English ā...","[{'name': 'inh', 'args': {'1': 'en', '2': 'enm...",0,10,-1,1
...,...,...,...,...,...,...,...,...
915346,iš,Lithuanian,"From Proto-Indo-European *eḱs (“out of”), *eǵʰ...","[{'name': 'der', 'args': {'1': 'lt', '2': 'ine...",0,9,-1,994
915347,stafróf,Icelandic,"From Old Norse stafróf, from Old English stæfr...","[{'name': 'der', 'args': {'1': 'is', '2': 'non...",0,9,-1,994
915348,fló,Old Norse,"table\nFrom Proto-Germanic *flauhaz, from Prot...","[{'name': 'etymid', 'args': {'1': 'non', '2': ...",2,9,-1,995
915349,échauffer,French,"Inherited from Middle French eschauffer, from ...","[{'name': 'glossary', 'args': {'1': 'Inherited...",0,9,-1,996


In [85]:
# def is_valid_lang(obj):
#     return langcodes.is_langcode(obj)
# is_valid_lang('en'), is_valid_lang('engllll') # (True, False)

(True, False)

In [163]:
template_frequencies = {}
for templates in df['templates']:
    for template in templates:
        # collect frequencies based on template['name']
        if template['name'] not in template_frequencies:
            template_frequencies[template['name']] = 0
        template_frequencies[template['name']] += 1

# sort template_frequncies by frequency
sorted_template_frequencies = sorted(template_frequencies.items(), key=lambda x: x[1], reverse=True)

In [27]:
# get top 100
sorted_template_frequencies[:10]

[('m', 633226),
 ('cog', 547420),
 ('inh', 431050),
 ('der', 390749),
 ('bor', 284354),
 ('glossary', 259118),
 ('bor+', 166964),
 ('inh+', 74002),
 ('af', 64558),
 ('root', 55225)]

In [120]:
sorted_template_frequencies[409] # everything with >= 20 occurrences

('vi-etym-hoi', 20)

In [137]:
# collate templates that belong to the top 100 most frequent templates
desired_templates = [x[0] for x in sorted_template_frequencies[:410]]
collated_templates = {x: [] for x in desired_templates}
for templates in df['templates']:
    for template in templates:
        if template['name'] in desired_templates:
            collated_templates[template['name']].append(template['args'])


In [164]:
def process_args(args):
    sr_by_key = {} # success rate by key
    for arg in args:
        for key, value in arg.items():
            if key not in sr_by_key:
                sr_by_key[key] = {'success': 0, 'total': 0}
            if value:
                sr_by_key[key]['total'] += 1
                if langcodes.is_langcode(value):
                    sr_by_key[key]['success'] += 1
    return sr_by_key

success_rates = {x: process_args(collated_templates[x]) for x in collated_templates}

In [165]:
{k: (f"n={v['total']}, " + f"{v['success'] / v['total']:.2%}" if v['total'] else "") for k, v in success_rates['m'].items()}

{'1': 'n=633226, 100.00%',
 '2': 'n=610204, 2.57%',
 '3': 'n=34831, 0.96%',
 '4': 'n=142390, 4.06%',
 't': 'n=77038, 3.40%',
 'tr': 'n=40919, 4.17%',
 'id': 'n=1674, 6.39%',
 'pos': 'n=10353, 0.56%',
 'lit': 'n=2772, 0.47%',
 'g': 'n=1634, 0.00%',
 'sc': 'n=1656, 0.00%',
 'ts': 'n=936, 3.74%',
 'gloss': 'n=13398, 3.52%',
 'g2': 'n=25, 0.00%',
 'g3': '',
 'g1': 'n=2, 0.00%'}

```
{'1': 'n=633226, 99.45%',
 '2': 'n=610204, 2.49%',
 '3': 'n=34831, 0.90%',
 '4': 'n=142390, 3.68%',
 't': 'n=77038, 3.18%',
 'tr': 'n=40919, 3.96%',
 'id': 'n=1674, 6.33%',
 'pos': 'n=10353, 0.56%',
 'lit': 'n=2772, 0.25%',
 'g': 'n=1634, 0.00%',
 'sc': 'n=1656, 0.00%',
 'ts': 'n=936, 3.74%',
 'gloss': 'n=13398, 3.16%',
 'g2': 'n=25, 0.00%',
 'g3': '',
 'g1': 'n=2, 0.00%'}
 ```

In [177]:
# a suspected lang key is any key that has a success rate of >95%
suspected_lang_keys_per_template = {}
_warning_keys = {}
for template, sr in success_rates.items():
    # criteria for being a suspected lang key
    # 1. key contains 'lang' OR
    # 2. success rate > 95% AND total > 5
    suspected_lang_keys = {k for k, v in sr.items() if 'lang' in k or v['total'] and v['total'] >= 5 and v['success'] / v['total'] >= 0.95}
    
    # check for those between 0.8 and 0.95 and print them as a warning: usually, 
    for k, v in sr.items():
        if v['total'] and v['total'] >= 5 and 0.5 <= v['success'] / v['total'] < .9999:
            # print(f"WARNING: {template} {k} {v['success'] / v['total']:.2%}")
            _warning_keys[(template, k)] = v['success'] / v['total']
    
    suspected_lang_keys_per_template[template] = suspected_lang_keys
    # print(template, suspected_lang_keys)
# print _warning_keys, in descending order
for x in reversed(sorted(_warning_keys.items(), key=lambda x: x[1])):
    print(f"WARNING: {x[0][0]} {x[0][1]} {x[1]:.2%}")


In [179]:
# view surprises
surprised = "cal"
for arg in collated_templates[surprised]: # 'l', 'm', 'surf'
    for key, value in arg.items():
        if key == '2':
            if not langcodes.is_langcode(value):
                print(reconstruct_template({"name": surprised, "args": arg}))



cal|en|nrf,frm,it|-|notext=


Exceptions: 
1. Han compound's 'ls' is one of ['i', 'ic', 'psc'] but do NOT represent langcodes
2. Same for liushu: ['p', 'i', 'ic', 'psc']

4. surface analysis & surf 1 can be a lang, but surf 1 can also be a reference to another template
(like +suf)
5. noncog contains a large frequency of language families, due to its migration from {{etyl}}
6. Hira-dakuten 2 is actually Hepburn romanization of the character, which happens to be a lot of used 2- or 3- letter codes
7. Kana-dakuten: same
8. wp & zh-wp's argument is technically a lang code, but not the exact same one https://en.wiktionary.org/wiki/Template:wikipedia
9. cog 1 may be a comma-separated list of langcodes. The same is true of bor 2, bor+ 2, 
(1 instance only:) clq 2, cal 2, ubor 2, ncog 1,
10. dercat may have "<" in fields normally occupied by a language.
11. inc-ext looks confusing
12. pi-root 1 and tl-bay sc are NOT languages


In [180]:
success_rates['lbor'] # inh 2 only has a 92% success rate
# der 2has an 89.4%
# lbor 2 has 89.44%

{'1': {'success': 6933, 'total': 6933},
 '2': {'success': 6933, 'total': 6933},
 '3': {'success': 13, 'total': 6928},
 't': {'success': 2, 'total': 532},
 '4': {'success': 0, 'total': 677},
 '5': {'success': 6, 'total': 460},
 'gloss': {'success': 0, 'total': 5},
 'notext': {'success': 0, 'total': 160},
 'nocap': {'success': 0, 'total': 215},
 'tr': {'success': 1, 'total': 331},
 'g': {'success': 0, 'total': 8},
 'pos': {'success': 0, 'total': 25},
 'lit': {'success': 0, 'total': 46},
 'nocat': {'success': 0, 'total': 6},
 'sort': {'success': 0, 'total': 30},
 'id': {'success': 0, 'total': 3},
 'alt': {'success': 0, 'total': 5},
 'ts': {'success': 0, 'total': 2}}

In [181]:
# write success_rates to file
import json
with open('template_langcode_proportions.json', 'w') as f:
    json.dump(success_rates, f)

In [182]:
suspected_lang_keys_per_template

{'m': {'1'},
 'cog': {'1'},
 'inh': {'1', '2'},
 'der': {'1', '2'},
 'bor': {'1', '2'},
 'glossary': set(),
 'bor+': {'1', '2'},
 'inh+': {'1', '2'},
 'af': {'1', 'lang1', 'lang2', 'lang3', 'lang4'},
 'root': {'1', '2'},
 'l': {'1'},
 'suffix': {'1', 'lang1', 'lang2'},
 'doublet': {'1'},
 'uder': {'1', '2'},
 'compound': {'1', 'lang1', 'lang2', 'lang3', 'lang4'},
 'dercat': {'1', '10', '2', '3', '4', '5', '6', '7', '8', '9'},
 'surf': {'1', 'lang1', 'lang2'},
 'm-g': set(),
 'lit': set(),
 'com': {'1',
  'lang1',
  'lang10',
  'lang2',
  'lang3',
  'lang4',
  'lang5',
  'lang6',
  'lang7',
  'lang8',
  'lang9'},
 'categorize': {'1'},
 'affix': {'1', 'lang1', 'lang2', 'lang3', 'lang4'},
 'vi-etym-sino': set(),
 'ko-etym-sino': set(),
 'prefix': {'1', 'lang1', 'lang2'},
 'suf': {'1', 'lang1', 'lang2'},
 'etydate': set(),
 'internationalism': {'1'},
 'etydate/the': set(),
 'zh-l': set(),
 'ja-r': set(),
 'lang': {'1'},
 'calque': {'1', '2'},
 'lbor': {'1', '2'},
 'ltc-l': set(),
 'm+': {'

In [32]:
collated_templates['m'][:30]

[{'1': 'en', '2': 'a-', '3': '', '4': 'on'},
 {'1': 'enm', '2': 'baften'},
 {'1': 'enm', '2': 'biaften'},
 {'1': 'ang', '2': 'be', '3': '', '4': 'by'},
 {'1': 'en', '2': 'by'},
 {'1': 'ang', '2': 'æftan', '3': '', '4': 'behind'},
 {'1': 'en', '2': 'after'},
 {'1': 'en', '2': 'aft'},
 {'1': 'en', '2': 'a-', '3': '', '4': 'on'},
 {'1': 'enm', '2': 'baften'},
 {'1': 'enm', '2': 'biaften'},
 {'1': 'ang', '2': 'be', '3': '', '4': 'by'},
 {'1': 'en', '2': 'by'},
 {'1': 'ang', '2': 'æftan', '3': '', '4': 'behind'},
 {'1': 'en', '2': 'after'},
 {'1': 'en', '2': 'aft'},
 {'1': 'enm', '2': 'abreiden', 't': 'to start up, awake, move, reproach'},
 {'1': 'gem-pro', '2': '*bregdaną', 't': 'to move, swing'},
 {'1': 'ine-pro', '2': '*bʰrēǵ-', 't': 'to shine'},
 {'1': 'ang', '2': 'ā-', '3': '', '4': 'away, out'},
 {'1': 'ang', '2': 'a-'},
 {'1': 'ang', '2': 'beran', '3': '', '4': 'to bear'},
 {'1': 'ang', '2': 'ā-', '3': '', '4': 'away, out'},
 {'1': 'ang', '2': 'a-'},
 {'1': 'ang', '2': 'beran', '3': 

In [186]:
multilingual = {k: v for k, v in suspected_lang_keys_per_template.items() if len(v) >= 2}

In [187]:
multilingual
# everything which looks like 'inh': {'1', '2'} we can assume that '1' is the source language and '2' is the target language

{'inh': {'1', '2'},
 'der': {'1', '2'},
 'bor': {'1', '2'},
 'bor+': {'1', '2'},
 'inh+': {'1', '2'},
 'af': {'1', 'lang1', 'lang2', 'lang3', 'lang4'},
 'root': {'1', '2'},
 'suffix': {'1', 'lang1', 'lang2'},
 'uder': {'1', '2'},
 'compound': {'1', 'lang1', 'lang2', 'lang3', 'lang4'},
 'dercat': {'1', '10', '2', '3', '4', '5', '6', '7', '8', '9'},
 'surf': {'1', 'lang1', 'lang2'},
 'com': {'1',
  'lang1',
  'lang10',
  'lang2',
  'lang3',
  'lang4',
  'lang5',
  'lang6',
  'lang7',
  'lang8',
  'lang9'},
 'affix': {'1', 'lang1', 'lang2', 'lang3', 'lang4'},
 'prefix': {'1', 'lang1', 'lang2'},
 'suf': {'1', 'lang1', 'lang2'},
 'calque': {'1', '2'},
 'lbor': {'1', '2'},
 'com+': {'1', 'lang1', 'lang2'},
 'cal': {'1', '2'},
 'der+': {'1', '2'},
 'blend': {'1', 'lang1', 'lang2'},
 'named-after': {'1', 'srclang'},
 'pre': {'1', 'lang1', 'lang2'},
 'confix': {'1', 'lang1', 'lang2'},
 'clq': {'1', '2'},
 'sl': {'1', '2'},
 'ubor': {'1', '2'},
 'obor': {'1', '2'},
 'pseudo-loan': {'1', '2'},
 '

In [ ]:
# recognizing that a key is for a word is trickier. Here are the criteria: 
# 1. template has a suspected lang key is NOT sufficient because the language could be encoded in the template (ie. Han compound)
# 2. relatively unique: high proportion of unique values
# 3. not "t=", "gloss=", "pos=", "g, g2, g3", "lit", "id"
# 4. high proportion of the suspected word is present in the expansion
# 5. that word is not in parenthesis (not the gloss)

Sample

In [110]:
# take every template with >10 occurences and print out an example
import clean_templates_util
importlib.reload(clean_templates_util)
from clean_templates_util import reconstruct_template
for template, args in collated_templates.items():
    if len(args) < 10:
        continue
    print('{{' + reconstruct_template({"name": template, "args": args[0]}) + '}}')

{{m|en|a-||on}}
{{cog|nl|breien|t=}}
{{inh|en|enm|abraiden}}
{{der|en|enm|baft}}
{{bor|en|fr|-}}
{{glossary|Inherited}}
{{bor+|it|ML.|armistitium}}
{{inh+|uk|zle-ort|казанъ|каза́нъ}}
{{af|pl|samo-|lot}}
{{root|en|ine-pro|*dewk-}}
{{l|ko|東國正韻 / 동국정운|tr=|t=|sc=|lit=|pos=}}
{{suffix|en|abduct|ion}}
{{doublet|fr|bâtonnée}}
{{uder|en|la|-}}
{{compound|ja|sort=|人|tr1=|t1=|と|tr2=|pos2=|形#Japanese:_nari-shape|alt3=|tr3=|t3=|pos3=}}
{{dercat|fr|gmh|gem-pro}}
{{surf|ru|война́|-ный}}
{{m-g|sow (<span class="Latn" lang="en">female pig</span>)}}
{{lit|what reason|nocap=}}
{{com|en|pine|tree}}
{{categorize|vi|Sino-Vietnamese words}}
{{affix|nl|be-|hoeven}}
{{vi-etym-sino|省}}
{{ko-etym-sino|煙|smoke|氣|gas}}
{{prefix|en|a|braid}}
{{suf|fi|metsä|o}}
{{etydate|1768|ref=}}
{{internationalism|id}}
{{etydate/the|1768}}
{{zh-l|太師}}
{{ja-r|連%用%形|れん%よう%けい|stem or continuative form}}
{{lang|okm|쎤〮}}
{{calque|en|eo|krokodili|nocap=}}
{{lbor|en|pi|Buddha|t=}}
{{ltc-l|羨|id=}}
{{m+|de|Schnur}}
{{noncog|es|huerco}}
